# Cake eating on a grid, function approximation and the consumption-savings model

Notes at <https://dse.iskh.me/cake-eating>.

## Solving on a grid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class cake_ongrid():
    '''Cake eating on a grid: the next period cake is chosen from the state grid'''

    def __init__(self, beta=.9, Wbar=10, ngrid=50):
        self.beta = beta                               # discount factor
        self.Wbar = Wbar                               # upper bound on cake size
        self.ngrid = ngrid                             # number of grid points
        self.epsilon = np.finfo(float).eps             # smallest positive float
        self.grid = np.linspace(self.epsilon, Wbar, ngrid)   # grid for both state and choice

    def bellman(self, V0):
        '''Bellman operator, V0 is a vector of values on the grid'''
        c = self.grid - self.grid[:, np.newaxis]       # states in columns, next cake in rows
        c[c == 0] = self.epsilon                       # avoid log(0) when nothing is eaten
        mask = c > 0                                   # feasible choices: eat a positive amount
        matV1 = np.full((self.ngrid, self.ngrid), -np.inf)   # -inf marks infeasible choices
        matV0 = np.repeat(V0.reshape(self.ngrid, 1), self.ngrid, 1)   # V0 of the next cake, in rows
        matV1[mask] = np.log(c[mask]) + self.beta * matV0[mask]       # maximand of the Bellman equation
        V1 = np.amax(matV1, axis=0)                    # maximum in every column
        c1 = self.grid - self.grid[np.argmax(matV1, axis=0)]          # optimal consumption
        return V1, c1

    def solve(self, maxiter=1000, tol=1e-4, callback=None):
        '''Value function iterations'''
        V0 = np.log(self.grid)                         # start from eating everything
        for i in range(maxiter):
            V1, c1 = self.bellman(V0)
            if callback: callback(iter=i, grid=self.grid, value=V1, policy=c1)
            if np.all(abs(V1 - V0) < tol):
                break
            V0 = V1
        else:
            raise RuntimeError('No convergence: maximum number of iterations achieved')
        return V1, c1

In [ ]:
def check_analytic(model, title=''):
    '''Plot the numerical solution of a cake eating model against the analytic one'''
    b = model.beta
    aV = lambda w: np.log(w)/(1-b) + np.log(1-b)/(1-b) + b*np.log(b)/(1-b)**2
    aP = lambda w: (1-b)*w
    xg = np.linspace(model.Wbar/1000, model.Wbar, 1000)   # dense grid for the analytic solution
    V, policy = model.solve()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.plot(model.grid[1:], V[1:], label='numerical')
    ax1.plot(xg, aV(xg), label='analytic')
    ax2.plot(model.grid, policy, label='numerical')
    ax2.plot(xg, aP(xg), label='analytic')
    for ax, t in ((ax1, 'Value function'), (ax2, 'Policy function')):
        ax.set(xlabel='Cake size, W', title=t)
        ax.grid(color='0.9')
        ax.legend()
    fig.suptitle(title)
    plt.show()
    return np.max(np.abs(policy - aP(model.grid)))

err = check_analytic(cake_ongrid(beta=0.9, Wbar=10, ngrid=100), 'Same grid for state and choice, 100 points')
print(f'Largest error in consumption: {err:.3f}')

## Polynomial approximation

In [ ]:
from scipy import interpolate
from numpy.polynomial import polynomial, chebyshev
import warnings

def approximations(f, a, b, n):
    '''Linear spline, cubic spline, polynomial on equal nodes and Chebyshev interpolation with n nodes'''
    xe = np.linspace(a, b, n)                          # equally spaced nodes
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')                # polyfit warns about poor conditioning
        p = polynomial.polyfit(xe, f(xe), n-1)
    return {'linear spline':           lambda x: np.interp(x, xe, f(xe)),
            'cubic spline':            interpolate.CubicSpline(xe, f(xe)),
            'polynomial, equal nodes': lambda x: polynomial.polyval(x, p),
            'Chebyshev, own nodes':               chebyshev.Chebyshev.interpolate(f, n-1, domain=[a, b])}

f = lambda x: np.exp(-x/4)*np.sin(x) + 1/(1 + x**2)   # function to approximate
a, b, n = -5, 10, 12                                   # interval and number of nodes
xd = np.linspace(a, b, 1000)                           # dense grid to measure errors
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 16))
ax1.plot(xd, f(xd), 'k', lw=2, label='f(x)')
ax1.scatter(np.linspace(a, b, n), f(np.linspace(a, b, n)), c='k', zorder=3)
for name, g in approximations(f, a, b, n).items():
    ax1.plot(xd, g(xd), label=name)
    ax2.semilogy(xd, np.abs(g(xd) - f(xd)) + 1e-16, label=name)
ax1.set(ylim=(-1.5, 2), title=f'{n} nodes')
ax2.set(ylim=(1e-6, 15), title='Absolute error')
for ax in (ax1, ax2):
    ax.legend(fontsize=8); ax.grid(color='0.9')
plt.show()

## Discretized choice with interpolation

In [ ]:
class cake_discretized():
    '''Cake eating with a separate choice grid and interpolated value function'''

    def __init__(self, beta=.9, Wbar=10, ngrid=50, nchgrid=100):
        self.beta = beta                               # discount factor
        self.Wbar = Wbar                               # upper bound on cake size
        self.ngrid = ngrid                             # number of points on the state grid
        self.nchgrid = nchgrid                         # number of points on the choice grid
        self.epsilon = np.finfo(float).eps             # smallest positive float
        self.grid = np.linspace(self.epsilon, Wbar, ngrid)   # grid for the state
        self.share = np.linspace(self.epsilon, 1, nchgrid)   # choices as shares of the cake

    def bellman(self, V0):
        '''Bellman operator, V0 is a vector of values on the state grid'''
        c = self.share[:, np.newaxis] * self.grid      # choices in rows, scaled to [0,W] in each column
        W1 = self.grid - c                             # next period cake, off the grid in general
        matV1 = np.log(c) + self.beta * np.interp(W1, self.grid, V0)   # linear interpolation of V0
        i = np.argmax(matV1, axis=0)                   # optimal choice in every column
        return matV1[i, np.arange(self.ngrid)], c[i, np.arange(self.ngrid)]

    solve = cake_ongrid.solve                          # the same value function iterations

In [ ]:
err = check_analytic(cake_discretized(beta=0.9, Wbar=10, ngrid=100, nchgrid=100),
                     'Separate choice grid, 100 + 100 points')
print(f'Largest error in consumption: {err:.3f}')
print(f'{"state grid":>10} {"choice grid":>12} {"max error in c":>15}')
for ng, nch in ((50, 50), (100, 100), (100, 500), (500, 100), (500, 500)):
    m = cake_discretized(beta=0.9, Wbar=10, ngrid=ng, nchgrid=nch)
    V, c = m.solve()
    print(f'{ng:>10} {nch:>12} {np.max(np.abs(c - (1-m.beta)*m.grid)):>15.4f}')